# Islamic Audiobook Generator
### Run each cell in order. Make sure Runtime > Change runtime type > T4 GPU is selected.

In [ ]:
# Step 1: Install dependencies
!pip install -q pymupdf pydub gradio coqui-tts

In [ ]:
# Step 2: Check GPU
!nvidia-smi

In [ ]:
# Step 3: Run the app
import os
import uuid
import fitz
import gradio as gr
from TTS.api import TTS
from pydub import AudioSegment

os.makedirs('uploads', exist_ok=True)
os.makedirs('outputs', exist_ok=True)
os.makedirs('voice_samples', exist_ok=True)

tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to('cuda')

def extract_text(pdf_path):
    text = ''
    doc = fitz.open(pdf_path)
    for page in doc:
        text += page.get_text('text') or ''
    return text

def chunk_text(text, max_chars=200):
    sentences = text.replace('\n', ' ').split('. ')
    chunks, current = [], ''
    for s in sentences:
        if len(current) + len(s) < max_chars:
            current += s + '. '
        else:
            if current:
                chunks.append(current.strip())
            current = s + '. '
    if current:
        chunks.append(current.strip())
    return chunks

def generate(pdf_file, voice_file):
    job_id = str(uuid.uuid4())
    voice_path = f'voice_samples/{job_id}.wav'
    output_path = f'outputs/{job_id}_audiobook.mp3'

    import shutil
    shutil.copy(voice_file, voice_path)

    text = extract_text(pdf_file)
    if not text.strip():
        return None, 'Could not extract text from PDF'

    chunks = chunk_text(text)
    audio_segments = []

    for i, chunk in enumerate(chunks):
        chunk_path = f'outputs/{job_id}_chunk_{i}.wav'
        tts.tts_to_file(text=chunk, speaker_wav=voice_path, language='ur', file_path=chunk_path)
        audio_segments.append(AudioSegment.from_wav(chunk_path))

    combined = sum(audio_segments)
    combined.export(output_path, format='mp3')

    for i in range(len(chunks)):
        os.remove(f'outputs/{job_id}_chunk_{i}.wav')

    return output_path, 'Done!'

with gr.Blocks(title='Islamic Audiobook Generator') as demo:
    gr.Markdown('# Islamic Audiobook Generator')
    gr.Markdown('Upload an Urdu Islamic PDF and a scholar voice sample (WAV, min 6 sec) to generate an audiobook.')
    with gr.Row():
        pdf_input = gr.File(label='Upload Islamic Book (PDF)', file_types=['.pdf'])
        voice_input = gr.File(label='Upload Scholar Voice Sample (WAV)', file_types=['.wav'])
    btn = gr.Button('Generate Audiobook', variant='primary')
    audio_output = gr.Audio(label='Generated Audiobook', type='filepath')
    status = gr.Textbox(label='Status')
    btn.click(fn=generate, inputs=[pdf_input, voice_input], outputs=[audio_output, status])

demo.launch(share=True)